# ANN (Artificial Neural Network) - Simple Example
**Story (retail shop):** We want to guess if a customer will **buy (1)** or **not buy (0)**.

**Inputs (x):**
- `x1` = time in shop (scaled 0 to 1)
- `x2` = discount given (scaled 0 to 1)

**Output (y):** 1 = buy, 0 = not buy

**Network shape:** 2 inputs -> 2 hidden neurons -> 1 output

## Step 1: Import and make data
We use only `numpy`. Each row is one customer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

X = np.array([[0.1, 0.1],
              [0.2, 0.9],
              [0.8, 0.2],
              [0.9, 0.9],
              [0.3, 0.8],
              [0.7, 0.1]])
y = np.array([[0],[1],[0],[1],[1],[0]])   # 1 = buy
print("X shape:", X.shape, " y shape:", y.shape)

## Step 2: Important functions
- **Sigmoid** changes any number into a value between 0 and 1: `1 / (1 + e^-z)`
- **Loss** tells how wrong we are: `1/2 * (predicted - actual)^2`

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(a):      # a = sigmoid output
    return a * (1 - a)

def loss(pred, actual):
    return 0.5 * np.mean((pred - actual) ** 2)

## Step 3: Start with random weights
Weights `W` = how strong a connection is. Bias `b` = a small shift.
At the start they are random, so the network is wrong.

In [ ]:
np.random.seed(1)
W1 = np.random.randn(2, 2)   # input -> hidden
b1 = np.zeros((1, 2))
W2 = np.random.randn(2, 1)   # hidden -> output
b2 = np.zeros((1, 1))
lr = 0.5                     # learning rate = size of each step

## Step 4: Train (the most important part)
Every round (epoch) has 3 parts:
1. **Forward pass** - make a prediction
2. **Backpropagation** - find how much each weight caused the error
3. **Gradient descent** - change each weight a little to reduce error:
   `new_weight = old_weight - lr * gradient`

In [ ]:
history = []
for epoch in range(5000):
    # 1) Forward pass
    h = sigmoid(X @ W1 + b1)        # hidden layer
    out = sigmoid(h @ W2 + b2)      # output layer

    # 2) Backpropagation
    d_out = (out - y) * sigmoid_derivative(out)
    d_h   = (d_out @ W2.T) * sigmoid_derivative(h)

    # 3) Gradient descent (update weights)
    W2 -= lr * (h.T @ d_out) / len(X)
    b2 -= lr * d_out.mean(axis=0, keepdims=True)
    W1 -= lr * (X.T @ d_h) / len(X)
    b1 -= lr * d_h.mean(axis=0, keepdims=True)

    history.append(loss(out, y))
    if epoch % 1000 == 0:
        print(f"Epoch {epoch:4d}  loss = {history[-1]:.4f}")

## Step 5: See the loss going down
A good network = loss goes down.

In [ ]:
plt.plot(history, color="tab:orange")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Training loss")
plt.show()

## Step 6: Test the network

In [ ]:
def predict(x):
    h = sigmoid(np.array(x) @ W1 + b1)
    return sigmoid(h @ W2 + b2)

print("Predictions (close to 1 = will buy):")
for row, real in zip(X, y):
    print(row, "->", round(predict(row).item(), 3), "| real:", int(real[0]))

new_customer = [0.5, 0.85]
p = predict(new_customer).item()
print("\nNew customer", new_customer, "-> buy chance:", round(p, 2),
      "=> BUY" if p > 0.5 else "=> NOT BUY")

## Bonus: Same network with Keras (short way)
Libraries like Keras do all the math above for you. Run this only if TensorFlow is installed (`pip install tensorflow`).

In [ ]:
try:
    from tensorflow import keras
    model = keras.Sequential([
        keras.layers.Input(shape=(2,)),
        keras.layers.Dense(2, activation="sigmoid"),
        keras.layers.Dense(1, activation="sigmoid"),
    ])
    model.compile(optimizer=keras.optimizers.SGD(0.5), loss="mse")
    model.fit(X, y, epochs=2000, verbose=0)
    print(model.predict(X, verbose=0).round(3))
except ImportError:
    print("TensorFlow not installed - skip this cell.")

## Summary
- Weights start random -> predictions are bad
- Forward pass -> loss -> backpropagation -> update weights
- Repeat many times -> loss gets small -> network learns